# Module 2: Connecting to and Extracting Data from Multiple Sources

**Unit A · Week 2** · Track 1 — Data Integration, Standards, Metadata & Quality

Introduces this track's real, three-source practice dataset: a SQL-style extraction from a local database standing in for the hub's ERP system, and a REST/HDX-style pull, run against files instead of the live internet so the notebook is reliable to re-run anywhere. Closes with the reverse pattern — publishing a hub's own dataset as a small API.

## Learning objectives

- Query a relational database with SQL from Python and R.
- Parse a REST/JSON-shaped extract the way an HDX or HAPI pull would arrive.
- Handle extraction failure gracefully instead of letting a script fail silently.


## Setup

This notebook reads the raw practice files in `../../../data/raw/`, built by
`data/make_track1_sources.py` — three partner-style exports shaped like a
real regional hub's source-system landscape (an ERP/financial export, a
survey-platform export, and an HDX-style pull), plus the P-code gazetteer
used to reconcile them. All four are **synthetic**; see `data/README.md`.
Run `python3 data/make_track1_sources.py` once from the repo root before
working through this notebook if those files aren't there yet.

## Lesson content

- **SQL basics for extraction.** `SELECT`, `WHERE`, enough to pull exactly the slice of a database table a hub needs, run from Python via `sqlite3`.
- **Calling REST APIs / pulling CODs from HDX.** In production this is `requests` against a live endpoint (HDX Python API client, or direct HAPI REST calls); this notebook reads the equivalent already-downloaded JSON/CSV shape so it stays reproducible offline.
- **Handling failure gracefully.** Timeouts, rate limits, and expired credentials are the norm once extraction is automated — always log and surface failures rather than let a script fail silently.

In [1]:
import sqlite3
import pandas as pd

# --- SQL extraction, simulating the hub's ERP/financial system ---
partner_a = pd.read_csv("../../../data/raw/partner_a_finance_export.csv")
conn = sqlite3.connect(":memory:")
partner_a.to_sql("finance_export", conn, index=False)

db_df = pd.read_sql(
    "SELECT reg, rev, date FROM finance_export WHERE rev >= ?",
    conn, params=[30.0],
)
conn.close()
print(f"{len(db_df):,} rows extracted via SQL (rev >= 30.0)")
db_df.head()

240 rows extracted via SQL (rev >= 30.0)


,reg,rev,date
0,Gasabo,39.37,01/01/2023
1,Gasabo,45.24,01/02/2023
2,Gasabo,41.16,01/03/2023
3,Gasabo,42.91,01/04/2023
4,Gasabo,42.05,01/05/2023


In [2]:
# --- "API" extraction, simulating an HDX/HAPI-style pull ---
# In production: requests.get(HAPI_URL, params=..., timeout=30).json()
# Here: the equivalent already-pulled shape, read the same defensive way.
def fetch_hdx_style(path, timeout_ok=True):
    if not timeout_ok:
        raise TimeoutError("simulated HDX endpoint timeout")
    return pd.read_csv(path)

try:
    api_df = fetch_hdx_style("../../../data/raw/partner_c_hdx_pull.csv")
    print(f"{len(api_df):,} rows extracted via the HDX-style pull")
except TimeoutError as e:
    print(f"Extraction failed, logged for retry: {e}")   # never fail silently
    api_df = pd.DataFrame()

api_df.head()

240 rows extracted via the HDX-style pull


,district_pcode,date,indicator,outcome
0,SIM-GAS,2023-01-01,sample_wellbeing_index,1
1,SIM-GAS,2023-02-01,sample_wellbeing_index,1
2,SIM-GAS,2023-03-01,sample_wellbeing_index,0
3,SIM-GAS,2023-04-01,sample_wellbeing_index,0
4,SIM-GAS,2023-05-01,sample_wellbeing_index,0


### Module 2 Extension — worked example: serving integrated data as an API

*Optional advanced pattern, completes the loop with Module 9's "publish" step.* The rest of this module covers
consuming other people's APIs; the reverse is exposing the hub's own harmonized, validated dataset (Module 9's
output) as a small REST API with **FastAPI**, so a Track 2 dashboard, a partner system, or a colleague's
notebook can request exactly the slice it needs, always the latest published version, instead of everyone
keeping their own downloaded CSV that quickly goes stale. This code is illustrative, it starts a server, so it
is shown here rather than executed inline; save it as `api.py` and run it from a terminal with `uvicorn`.

```python
# api.py -- run with: uvicorn api:app --reload
from fastapi import FastAPI, HTTPException
import pandas as pd

app = FastAPI(title="Regional Hub Data API", version="1.0")

def load_latest():
    return pd.read_csv("../../../data/processed/track2_dataset.csv")

@app.get("/districts")
def list_districts():
    df = load_latest()
    return df[["district_pcode", "district"]].drop_duplicates().to_dict(orient="records")

@app.get("/districts/{pcode}")
def get_district(pcode: str):
    df = load_latest()
    result = df[df["district_pcode"] == pcode]
    if result.empty:
        raise HTTPException(status_code=404, detail="P-code not found: " + pcode)
    return result.to_dict(orient="records")

# Interactive docs are generated automatically at /docs
```

## Your turn

Extract the finance export via a SQL query and the HDX-style pull via the file-based simulation above, in both Python and R, and confirm the failure-handling branch logs rather than crashes when `timeout_ok=False`.

**Formative assessment.** Submitted script plus resulting extracted tables, graded on correct filtering, and on graceful (not silent) handling of the simulated timeout.

## Governance / responsibility callback

The discussion prompt for the API extension: what access control would this API need before it could be exposed outside the hub's internal network — and which Module 10 governance practices (licensing, sensitive-field review, an accountable owner) apply just as much to an API endpoint as to a shared CSV file?